# Home Credit Default Risk  
## Data Merging and Feature Engineering

This notebook:

1. Loads the Home Credit competition tables.
2. Engineers features from the current application.
3. Aggregates historical tables to one row per `SK_ID_CURR`.
4. Merges all tables into final train and test dataframes.
5. Performs basic validation and saves the results.

The implementation is designed to be reasonably friendly to free Google Colab environments.

> Update `DATA_DIR` before running the notebook.


In [50]:
import pandas as pd

manifest = pd.read_csv("demo_cases/manifest.csv")

medium_case = (
    manifest[manifest["risk_category"] == "medium"]
    .sort_values("default_probability", ascending=False)
    .iloc[0]
)

low_case = (
    manifest[manifest["risk_category"] == "low"]
    .sort_values("default_probability", ascending=False)
    .iloc[0]
)

print("Medium case:")
print(medium_case)

print("\nLow case:")
print(low_case)

Medium case:
demo_case_id           DEMO-005
actual_default                1
default_probability    0.504279
risk_category            medium
Name: 4, dtype: object

Low case:
demo_case_id           DEMO-014
actual_default                0
default_probability    0.295787
risk_category               low
Name: 13, dtype: object


In [51]:
import json

from app.predictor import predict_default_probability


with open(
    "sample_requests/sample_request_raw.json",
    "r",
    encoding="utf-8"
) as file:
    payload = json.load(file)


results = predict_default_probability(payload["records"])

print(results)


# Basic validation
assert len(results) == len(payload["records"])

for result in results:
    probability = result["default_probability"]

    assert 0 <= probability <= 1

print("End-to-end prediction test passed.")


[{'default_probability': 0.756365954875946, 'non_default_probability': 0.24363404512405396, 'risk_category': 'high', 'model_version': '1.1.0'}]
End-to-end prediction test passed.


In [52]:
import json

with open(
    "demo_cases/DEMO-014/report_input.json",
    encoding="utf-8",
) as file:
    data = json.load(file)

evaluation = data["deterministic_policy_evaluation"]

print(evaluation["summary"])

for rule in evaluation["rules"]:
    print(
        rule["rule_id"],
        "->",
        rule["evaluation_status"],
        "|",
        rule["reason"],
    )

{'recommendation': 'manual_review', 'human_review_required': True, 'failed_rule_ids': [], 'manual_review_rule_ids': [], 'unknown_required_rule_ids': ['CREDIT_TO_INCOME', 'FOIR', 'REPAYMENT_HISTORY_DPD', 'KYC_AND_CONSENT', 'DOCUMENT_COMPLETENESS']}
MODEL_RISK_BAND -> pass | The applicant is in the low model-risk band. Straight-through processing still requires all other mandatory policy checks to pass.
CREDIT_TO_INCOME -> unknown | The reported ratio is 14.834, but verified annual income has not been confirmed. The policy threshold cannot be deterministically applied.
FOIR -> unknown | FOIR is unavailable.
REPAYMENT_HISTORY_DPD -> unknown | The required six-month and twelve-month DPD history is incomplete.
KYC_AND_CONSENT -> unknown | KYC verification and valid consent are not both confirmed.
DOCUMENT_COMPLETENESS -> unknown | Document completeness is not available.


In [53]:
import json

from app.predictor import predict_default_probability


with open(
    "sample_requests/sample_request_raw.json",
    encoding="utf-8",
) as file:
    payload = json.load(file)

results = predict_default_probability(payload["records"])

print(results)

assert results[0]["risk_category"] == "high"
assert results[0]["model_version"] == "1.1.0"
#assert "applicant_id" in results[0]


[{'default_probability': 0.756365954875946, 'non_default_probability': 0.24363404512405396, 'risk_category': 'high', 'model_version': '1.1.0'}]


In [54]:
results = predict_default_probability(payload["records"])

print(results)

assert results[0]["risk_category"] == "high"
assert results[0]["model_version"] == "1.1.0"


[{'default_probability': 0.756365954875946, 'non_default_probability': 0.24363404512405396, 'risk_category': 'high', 'model_version': '1.1.0'}]


In [55]:
import json
from pathlib import Path

import joblib
import pandas as pd


BUNDLE_PATH = Path("artifacts/home_credit_bundle.joblib")
INPUT_PATH = Path("sample_requests/sample_request.json")
OUTPUT_PATH = Path("sample_requests/sample_request_raw.json")


# Load saved preprocessing objects
bundle = joblib.load(BUNDLE_PATH)

encoder = bundle["ordinal_encoder"]
categorical_cols = list(bundle["categorical_cols"])


# Load the currently encoded request
with open(INPUT_PATH, "r", encoding="utf-8") as file:
    payload = json.load(file)

df = pd.DataFrame(payload["records"])


# Verify all categorical columns are available
missing_cols = [
    col for col in categorical_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(
        f"Missing categorical columns: {missing_cols}"
    )


# Convert encoded numbers back to original string categories
df[categorical_cols] = encoder.inverse_transform(
    df[categorical_cols]
)


# Convert DataFrame safely into JSON-compatible records
raw_records = json.loads(
    df.to_json(
        orient="records",
        date_format="iso"
    )
)


# Save the corrected request
output_payload = {
    "records": raw_records
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as file:
    json.dump(
        output_payload,
        file,
        indent=2,
        ensure_ascii=False,
        allow_nan=False
    )

print(f"Raw request saved to: {OUTPUT_PATH}")
print(df[categorical_cols].iloc[0])

Raw request saved to: sample_requests/sample_request_raw.json
NAME_CONTRACT_TYPE                 Cash loans
CODE_GENDER                                 M
FLAG_OWN_CAR                                N
FLAG_OWN_REALTY                             Y
NAME_TYPE_SUITE                 Unaccompanied
                                ...          
ORGANIZATION_TYPE      Business Entity Type 3
FONDKAPREMONT_MODE           reg oper account
HOUSETYPE_MODE                 block of flats
WALLSMATERIAL_MODE               Stone, brick
EMERGENCYSTATE_MODE                        No
Name: 0, Length: 16, dtype: object


In [56]:
import gc
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 10)

# Change this path when necessary.
DATA_DIR = Path("home_credit_data")

print("Data directory:", DATA_DIR)


Data directory: home_credit_data


In [57]:
import os
print(os.getcwd())

/Users/jayeshadwani/ml-learning/notebooks/Case Studies/Credit Risk Prediction/home_credit_risk_api


In [58]:
print(DATA_DIR)
print((DATA_DIR / "application_train.csv").is_file())

home_credit_data
True


## 1. Utility functions

The historical tables contain multiple rows per customer. Before merging them into the application table, each table must be aggregated to the `SK_ID_CURR` level.

Directly merging raw historical tables would create duplicate application rows.


In [59]:
def flatten_columns(df):
    """Flatten MultiIndex aggregation column names."""
    df = df.copy()

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            "_".join([str(part) for part in col if str(part) != ""]).upper()
            for col in df.columns
        ]
    else:
        df.columns = [str(col).upper() for col in df.columns]

    return df


def safe_divide(numerator, denominator):
    """Divide while preventing division-by-zero errors."""
    denominator = denominator.replace(0, np.nan)
    result = numerator / denominator
    return result.replace([np.inf, -np.inf], np.nan)


def reduce_memory_usage(df):
    """Downcast numeric columns to reduce RAM usage."""
    start_memory = df.memory_usage(deep=True).sum() / 1024**2

    for col in df.columns:
        col_type = df[col].dtype

        if pd.api.types.is_integer_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="integer")

        elif pd.api.types.is_float_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="float")

    end_memory = df.memory_usage(deep=True).sum() / 1024**2
    reduction = 100 * (start_memory - end_memory) / max(start_memory, 1e-9)

    print(
        f"Memory: {start_memory:.2f} MB -> "
        f"{end_memory:.2f} MB ({reduction:.1f}% reduction)"
    )

    return df


def load_csv(filename, usecols=None):
    """Load a CSV file from DATA_DIR."""
    path = DATA_DIR / filename

    if not path.exists():
        raise FileNotFoundError(
            f"{path} was not found. Check DATA_DIR and the extracted files."
        )

    df = pd.read_csv(path, usecols=usecols)
    print(f"{filename}: {df.shape}")
    return reduce_memory_usage(df)


## 2. Load the main application tables


In [60]:
application_train = load_csv("application_train.csv")
application_test = load_csv("application_test.csv")

print("Train shape:", application_train.shape)
print("Test shape:", application_test.shape)
print("Target distribution:")
display(application_train["TARGET"].value_counts(normalize=True).rename("proportion"))


application_train.csv: (307511, 122)


KeyboardInterrupt: 

In [ ]:
DATA_DIR = Path(
    "/Users/jayeshadwani/ml-learning/notebooks/Case Studies/"
    "Credit Risk Prediction/home_credit_risk_api/home_credit_data"
)

print("Data directory:", DATA_DIR)
print("Training data exists:", (DATA_DIR / "application_train.csv").exists())

Data directory: /Users/jayeshadwani/ml-learning/notebooks/Case Studies/Credit Risk Prediction/home_credit_risk_api/home_credit_data
Training data exists: True


## 3. Application-level feature engineering

These features are created directly from the current application:

- Age and employment duration
- Credit-to-income and annuity-to-income ratios
- Estimated credit term
- Family and child ratios
- External credit score summaries
- Contact information count
- Document count
- Housing summary statistics


In [ ]:
def engineer_application_features(df):
    df = df.copy()

    # Convert negative day offsets into positive years.
    df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365.25
    df["EMPLOYED_YEARS"] = -df["DAYS_EMPLOYED"] / 365.25
    df["REGISTRATION_YEARS"] = -df["DAYS_REGISTRATION"] / 365.25
    df["ID_PUBLISH_YEARS"] = -df["DAYS_ID_PUBLISH"] / 365.25

    # DAYS_EMPLOYED contains a known placeholder value of 365243.
    df["DAYS_EMPLOYED_ANOMALY"] = (df["DAYS_EMPLOYED"] == 365243).astype("int8")
    df.loc[df["DAYS_EMPLOYED"] == 365243, "DAYS_EMPLOYED"] = np.nan
    df.loc[df["EMPLOYED_YEARS"] < 0, "EMPLOYED_YEARS"] = np.nan

    # Core affordability and leverage ratios.
    df["CREDIT_INCOME_RATIO"] = safe_divide(
        df["AMT_CREDIT"], df["AMT_INCOME_TOTAL"]
    )
    df["ANNUITY_INCOME_RATIO"] = safe_divide(
        df["AMT_ANNUITY"], df["AMT_INCOME_TOTAL"]
    )
    df["GOODS_INCOME_RATIO"] = safe_divide(
        df["AMT_GOODS_PRICE"], df["AMT_INCOME_TOTAL"]
    )
    df["CREDIT_GOODS_RATIO"] = safe_divide(
        df["AMT_CREDIT"], df["AMT_GOODS_PRICE"]
    )
    df["CREDIT_ANNUITY_RATIO"] = safe_divide(
        df["AMT_CREDIT"], df["AMT_ANNUITY"]
    )

    # Family structure.
    df["INCOME_PER_PERSON"] = safe_divide(
        df["AMT_INCOME_TOTAL"], df["CNT_FAM_MEMBERS"]
    )
    df["CHILDREN_RATIO"] = safe_divide(
        df["CNT_CHILDREN"], df["CNT_FAM_MEMBERS"]
    )

    # Employment relative to age.
    df["EMPLOYED_AGE_RATIO"] = safe_divide(
        df["EMPLOYED_YEARS"], df["AGE_YEARS"]
    )

    # External credit-score summaries.
    ext_cols = [
        col for col in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
        if col in df.columns
    ]

    if ext_cols:
        df["EXT_SOURCE_MEAN"] = df[ext_cols].mean(axis=1)
        df["EXT_SOURCE_MIN"] = df[ext_cols].min(axis=1)
        df["EXT_SOURCE_MAX"] = df[ext_cols].max(axis=1)
        df["EXT_SOURCE_STD"] = df[ext_cols].std(axis=1)
        df["EXT_SOURCE_MISSING_COUNT"] = df[ext_cols].isna().sum(axis=1)

    # Contact availability.
    contact_cols = [
        col for col in [
            "FLAG_MOBIL",
            "FLAG_EMP_PHONE",
            "FLAG_WORK_PHONE",
            "FLAG_CONT_MOBILE",
            "FLAG_PHONE",
            "FLAG_EMAIL",
        ]
        if col in df.columns
    ]

    if contact_cols:
        df["CONTACT_FLAGS_SUM"] = df[contact_cols].sum(axis=1)

    # Document availability.
    document_cols = [
        col for col in df.columns if col.startswith("FLAG_DOCUMENT_")
    ]

    if document_cols:
        df["DOCUMENT_FLAGS_SUM"] = df[document_cols].sum(axis=1)

    # Housing summaries.
    housing_cols = [
        col for col in df.columns
        if col.endswith(("_AVG", "_MODE", "_MEDI"))
        and pd.api.types.is_numeric_dtype(df[col])
    ]

    if housing_cols:
        df["HOUSING_FEATURE_MEAN"] = df[housing_cols].mean(axis=1)
        df["HOUSING_FEATURE_MISSING_COUNT"] = (
            df[housing_cols].isna().sum(axis=1)
        )

    # Social-circle ratios.
    if {
        "DEF_30_CNT_SOCIAL_CIRCLE",
        "OBS_30_CNT_SOCIAL_CIRCLE",
    }.issubset(df.columns):
        df["SOCIAL_DEFAULT_30_RATIO"] = safe_divide(
            df["DEF_30_CNT_SOCIAL_CIRCLE"],
            df["OBS_30_CNT_SOCIAL_CIRCLE"],
        )

    if {
        "DEF_60_CNT_SOCIAL_CIRCLE",
        "OBS_60_CNT_SOCIAL_CIRCLE",
    }.issubset(df.columns):
        df["SOCIAL_DEFAULT_60_RATIO"] = safe_divide(
            df["DEF_60_CNT_SOCIAL_CIRCLE"],
            df["OBS_60_CNT_SOCIAL_CIRCLE"],
        )

    # Region and city mismatch count.
    mismatch_cols = [
        col for col in [
            "REG_REGION_NOT_LIVE_REGION",
            "REG_REGION_NOT_WORK_REGION",
            "LIVE_REGION_NOT_WORK_REGION",
            "REG_CITY_NOT_LIVE_CITY",
            "REG_CITY_NOT_WORK_CITY",
            "LIVE_CITY_NOT_WORK_CITY",
        ]
        if col in df.columns
    ]

    if mismatch_cols:
        df["LOCATION_MISMATCH_COUNT"] = df[mismatch_cols].sum(axis=1)

    return df


application_train = engineer_application_features(application_train)
application_test = engineer_application_features(application_test)

print("Engineered train shape:", application_train.shape)
print("Engineered test shape:", application_test.shape)


Engineered train shape: (307511, 147)
Engineered test shape: (48744, 146)


## 4. Bureau and bureau-balance features

`bureau.csv` contains credits from other financial institutions.

`bureau_balance.csv` contains monthly status information for each bureau credit. It is first aggregated by `SK_ID_BUREAU`, merged into `bureau.csv`, and then aggregated by `SK_ID_CURR`.


In [ ]:
def aggregate_bureau(data_dir):
    bureau = load_csv("bureau.csv")
    bureau_balance = load_csv("bureau_balance.csv")

    # Convert bureau monthly status into dummy variables.
    bb_status = pd.get_dummies(
        bureau_balance["STATUS"],
        prefix="BB_STATUS",
        dummy_na=True,
        dtype="uint8",
    )

    bureau_balance_encoded = pd.concat(
        [
            bureau_balance[["SK_ID_BUREAU", "MONTHS_BALANCE"]],
            bb_status,
        ],
        axis=1,
    )

    bb_agg_dict = {
        "MONTHS_BALANCE": ["min", "max", "mean", "count"],
    }

    for col in bb_status.columns:
        bb_agg_dict[col] = ["mean", "sum"]

    bb_agg = bureau_balance_encoded.groupby("SK_ID_BUREAU").agg(bb_agg_dict)
    bb_agg = flatten_columns(bb_agg).reset_index()

    bureau = bureau.merge(bb_agg, on="SK_ID_BUREAU", how="left")

    # Bureau categorical features.
    bureau_category_cols = [
        col for col in ["CREDIT_ACTIVE", "CREDIT_CURRENCY", "CREDIT_TYPE"]
        if col in bureau.columns
    ]

    bureau_encoded = pd.get_dummies(
        bureau,
        columns=bureau_category_cols,
        dummy_na=True,
        dtype="uint8",
    )

    # Additional bureau-level ratios.
    bureau_encoded["BUREAU_DEBT_CREDIT_RATIO"] = safe_divide(
        bureau_encoded["AMT_CREDIT_SUM_DEBT"],
        bureau_encoded["AMT_CREDIT_SUM"],
    )
    bureau_encoded["BUREAU_OVERDUE_CREDIT_RATIO"] = safe_divide(
        bureau_encoded["AMT_CREDIT_SUM_OVERDUE"],
        bureau_encoded["AMT_CREDIT_SUM"],
    )

    agg = {
        "SK_ID_BUREAU": ["count"],
        "DAYS_CREDIT": ["min", "max", "mean", "std"],
        "CREDIT_DAY_OVERDUE": ["max", "mean", "sum"],
        "DAYS_CREDIT_ENDDATE": ["min", "max", "mean"],
        "DAYS_ENDDATE_FACT": ["min", "max", "mean"],
        "AMT_CREDIT_MAX_OVERDUE": ["max", "mean"],
        "CNT_CREDIT_PROLONG": ["max", "mean", "sum"],
        "AMT_CREDIT_SUM": ["min", "max", "mean", "sum"],
        "AMT_CREDIT_SUM_DEBT": ["min", "max", "mean", "sum"],
        "AMT_CREDIT_SUM_LIMIT": ["min", "max", "mean", "sum"],
        "AMT_CREDIT_SUM_OVERDUE": ["max", "mean", "sum"],
        "AMT_ANNUITY": ["max", "mean", "sum"],
        "BUREAU_DEBT_CREDIT_RATIO": ["mean", "max"],
        "BUREAU_OVERDUE_CREDIT_RATIO": ["mean", "max"],
    }

    # Add all generated dummy and bureau-balance columns.
    additional_cols = [
        col for col in bureau_encoded.columns
        if col.startswith(("CREDIT_ACTIVE_", "CREDIT_TYPE_", "CREDIT_CURRENCY_", "BB_STATUS_"))
    ]

    for col in additional_cols:
        agg[col] = ["mean", "sum"]

    # Only retain columns that exist.
    agg = {
        col: funcs
        for col, funcs in agg.items()
        if col in bureau_encoded.columns
    }

    bureau_agg = bureau_encoded.groupby("SK_ID_CURR").agg(agg)
    bureau_agg = flatten_columns(bureau_agg).reset_index()
    bureau_agg = bureau_agg.rename(
        columns={
            col: f"BUREAU_{col}"
            for col in bureau_agg.columns
            if col != "SK_ID_CURR"
        }
    )

    del bureau, bureau_balance, bureau_balance_encoded, bureau_encoded, bb_agg
    gc.collect()

    return reduce_memory_usage(bureau_agg)


bureau_agg = aggregate_bureau(DATA_DIR)
print("Bureau aggregate shape:", bureau_agg.shape)
display(bureau_agg.head())


bureau.csv: (1716428, 17)
Memory: 512.11 MB -> 448.27 MB (12.5% reduction)
bureau_balance.csv: (27299925, 3)
Memory: 1926.61 MB -> 1640.22 MB (14.9% reduction)
Memory: 230.98 MB -> 147.28 MB (36.2% reduction)
Bureau aggregate shape: (305811, 130)


,SK_ID_CURR,BUREAU_SK_ID_BUREAU_COUNT,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_STD,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_CREDIT_DAY_OVERDUE_SUM,BUREAU_DAYS_CREDIT_ENDDATE_MIN,...,BUREAU_CREDIT_TYPE_MOBILE OPERATOR LOAN_MEAN,BUREAU_CREDIT_TYPE_MOBILE OPERATOR LOAN_SUM,BUREAU_CREDIT_TYPE_MORTGAGE_MEAN,BUREAU_CREDIT_TYPE_MORTGAGE_SUM,BUREAU_CREDIT_TYPE_REAL ESTATE LOAN_MEAN,BUREAU_CREDIT_TYPE_REAL ESTATE LOAN_SUM,BUREAU_CREDIT_TYPE_UNKNOWN TYPE OF LOAN_MEAN,BUREAU_CREDIT_TYPE_UNKNOWN TYPE OF LOAN_SUM,BUREAU_CREDIT_TYPE_NAN_MEAN,BUREAU_CREDIT_TYPE_NAN_SUM
0,100001,7,-1572,-49,-735.000000,489.942505,0,0.0,0,-1329.0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
1,100002,8,-1437,-103,-874.000000,431.451050,0,0.0,0,-1072.0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
2,100003,4,-2586,-606,-1400.750000,909.826111,0,0.0,0,-2434.0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
3,100004,2,-1326,-408,-867.000000,649.124023,0,0.0,0,-595.0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
4,100005,3,-373,-62,-190.666672,162.297058,0,0.0,0,-128.0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0


## 5. Previous-application features

This table contains earlier applications made by the same customer to Home Credit.


In [ ]:
def aggregate_previous_applications():
    previous = load_csv("previous_application.csv")

    # Known placeholder for missing day values.
    day_cols = [
        col for col in previous.columns
        if col.startswith("DAYS_")
    ]

    for col in day_cols:
        previous[col] = previous[col].replace(365243, np.nan)

    previous["PREV_APPLICATION_CREDIT_RATIO"] = safe_divide(
        previous["AMT_APPLICATION"],
        previous["AMT_CREDIT"],
    )
    previous["PREV_CREDIT_ANNUITY_RATIO"] = safe_divide(
        previous["AMT_CREDIT"],
        previous["AMT_ANNUITY"],
    )
    previous["PREV_DOWN_PAYMENT_RATIO"] = safe_divide(
        previous["AMT_DOWN_PAYMENT"],
        previous["AMT_CREDIT"],
    )

    category_cols = previous.select_dtypes(include=["object", "category"]).columns.tolist()

    previous_encoded = pd.get_dummies(
        previous,
        columns=category_cols,
        dummy_na=True,
        dtype="uint8",
    )

    agg = {
        "SK_ID_PREV": ["count"],
        "AMT_ANNUITY": ["min", "max", "mean", "sum"],
        "AMT_APPLICATION": ["min", "max", "mean", "sum"],
        "AMT_CREDIT": ["min", "max", "mean", "sum"],
        "AMT_DOWN_PAYMENT": ["min", "max", "mean", "sum"],
        "AMT_GOODS_PRICE": ["min", "max", "mean"],
        "HOUR_APPR_PROCESS_START": ["min", "max", "mean"],
        "RATE_DOWN_PAYMENT": ["min", "max", "mean"],
        "DAYS_DECISION": ["min", "max", "mean"],
        "CNT_PAYMENT": ["min", "max", "mean", "sum"],
        "PREV_APPLICATION_CREDIT_RATIO": ["mean", "max"],
        "PREV_CREDIT_ANNUITY_RATIO": ["mean", "max"],
        "PREV_DOWN_PAYMENT_RATIO": ["mean", "max"],
    }

    dummy_cols = [
        col for col in previous_encoded.columns
        if col not in previous.columns
    ]

    for col in dummy_cols:
        agg[col] = ["mean", "sum"]

    agg = {
        col: funcs
        for col, funcs in agg.items()
        if col in previous_encoded.columns
    }

    previous_agg = previous_encoded.groupby("SK_ID_CURR").agg(agg)
    previous_agg = flatten_columns(previous_agg).reset_index()
    previous_agg = previous_agg.rename(
        columns={
            col: f"PREV_{col}"
            for col in previous_agg.columns
            if col != "SK_ID_CURR"
        }
    )

    del previous, previous_encoded
    gc.collect()

    return reduce_memory_usage(previous_agg)


previous_agg = aggregate_previous_applications()
print("Previous application aggregate shape:", previous_agg.shape)
display(previous_agg.head())


previous_application.csv: (1670214, 37)
Memory: 1900.63 MB -> 1785.95 MB (6.0% reduction)
Memory: 547.11 MB -> 328.98 MB (39.9% reduction)
Previous application aggregate shape: (338857, 358)


,SK_ID_CURR,PREV_SK_ID_PREV_COUNT,PREV_AMT_ANNUITY_MIN,PREV_AMT_ANNUITY_MAX,PREV_AMT_ANNUITY_MEAN,PREV_AMT_ANNUITY_SUM,PREV_AMT_APPLICATION_MIN,PREV_AMT_APPLICATION_MAX,PREV_AMT_APPLICATION_MEAN,PREV_AMT_APPLICATION_SUM,...,PREV_PRODUCT_COMBINATION_POS MOBILE WITH INTEREST_MEAN,PREV_PRODUCT_COMBINATION_POS MOBILE WITH INTEREST_SUM,PREV_PRODUCT_COMBINATION_POS MOBILE WITHOUT INTEREST_MEAN,PREV_PRODUCT_COMBINATION_POS MOBILE WITHOUT INTEREST_SUM,PREV_PRODUCT_COMBINATION_POS OTHER WITH INTEREST_MEAN,PREV_PRODUCT_COMBINATION_POS OTHER WITH INTEREST_SUM,PREV_PRODUCT_COMBINATION_POS OTHERS WITHOUT INTEREST_MEAN,PREV_PRODUCT_COMBINATION_POS OTHERS WITHOUT INTEREST_SUM,PREV_PRODUCT_COMBINATION_NAN_MEAN,PREV_PRODUCT_COMBINATION_NAN_SUM
0,100001,1,3951.000,3951.000,3951.000,3951.000,24835.5,24835.5,24835.50,24835.5,...,1.0,1,0.0,0,0.0,0,0.0,0,0.0,0
1,100002,1,9251.775,9251.775,9251.775,9251.775,179055.0,179055.0,179055.00,179055.0,...,0.0,0,0.0,0,1.0,1,0.0,0,0.0,0
2,100003,3,6737.310,98356.995,56553.990,169661.970,68809.5,900000.0,435436.50,1306309.5,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
3,100004,1,5357.250,5357.250,5357.250,5357.250,24282.0,24282.0,24282.00,24282.0,...,0.0,0,1.0,1,0.0,0,0.0,0,0.0,0
4,100005,2,4813.200,4813.200,4813.200,4813.200,0.0,44617.5,22308.75,44617.5,...,0.5,1,0.0,0,0.0,0,0.0,0,0.0,0


## 6. Installment-payment features

Important engineered variables:

- `PAYMENT_DIFFERENCE`: scheduled installment minus actual payment
- `PAYMENT_RATIO`: actual payment divided by scheduled installment
- `DPD`: days past due
- `DBD`: days paid before due date


In [ ]:
def aggregate_installments():
    installments = load_csv("installments_payments.csv")

    installments["PAYMENT_DIFFERENCE"] = (
        installments["AMT_INSTALMENT"] - installments["AMT_PAYMENT"]
    )

    installments["PAYMENT_RATIO"] = safe_divide(
        installments["AMT_PAYMENT"],
        installments["AMT_INSTALMENT"],
    )

    # Positive values mean the payment was late.
    installments["DPD"] = (
        installments["DAYS_ENTRY_PAYMENT"] - installments["DAYS_INSTALMENT"]
    ).clip(lower=0)

    # Positive values mean the payment was made early.
    installments["DBD"] = (
        installments["DAYS_INSTALMENT"] - installments["DAYS_ENTRY_PAYMENT"]
    ).clip(lower=0)

    installments["LATE_PAYMENT_FLAG"] = (
        installments["DPD"] > 0
    ).astype("uint8")

    installments["UNDERPAYMENT_FLAG"] = (
        installments["AMT_PAYMENT"] < installments["AMT_INSTALMENT"]
    ).astype("uint8")

    agg = {
        "SK_ID_PREV": ["nunique"],
        "NUM_INSTALMENT_VERSION": ["nunique"],
        "NUM_INSTALMENT_NUMBER": ["max", "mean"],
        "DAYS_INSTALMENT": ["min", "max", "mean"],
        "DAYS_ENTRY_PAYMENT": ["min", "max", "mean"],
        "AMT_INSTALMENT": ["min", "max", "mean", "sum"],
        "AMT_PAYMENT": ["min", "max", "mean", "sum"],
        "PAYMENT_DIFFERENCE": ["min", "max", "mean", "sum"],
        "PAYMENT_RATIO": ["min", "max", "mean"],
        "DPD": ["max", "mean", "sum"],
        "DBD": ["max", "mean", "sum"],
        "LATE_PAYMENT_FLAG": ["mean", "sum"],
        "UNDERPAYMENT_FLAG": ["mean", "sum"],
    }

    installments_agg = installments.groupby("SK_ID_CURR").agg(agg)
    installments_agg = flatten_columns(installments_agg).reset_index()
    installments_agg = installments_agg.rename(
        columns={
            col: f"INST_{col}"
            for col in installments_agg.columns
            if col != "SK_ID_CURR"
        }
    )

    del installments
    gc.collect()

    return reduce_memory_usage(installments_agg)


installments_agg = aggregate_installments()
print("Installments aggregate shape:", installments_agg.shape)
display(installments_agg.head())


installments_payments.csv: (13605401, 8)
Memory: 830.41 MB -> 493.05 MB (40.6% reduction)
Memory: 69.95 MB -> 58.94 MB (15.7% reduction)
Installments aggregate shape: (339587, 36)


,SK_ID_CURR,INST_SK_ID_PREV_NUNIQUE,INST_NUM_INSTALMENT_VERSION_NUNIQUE,INST_NUM_INSTALMENT_NUMBER_MAX,INST_NUM_INSTALMENT_NUMBER_MEAN,INST_DAYS_INSTALMENT_MIN,INST_DAYS_INSTALMENT_MAX,INST_DAYS_INSTALMENT_MEAN,INST_DAYS_ENTRY_PAYMENT_MIN,INST_DAYS_ENTRY_PAYMENT_MAX,...,INST_DPD_MAX,INST_DPD_MEAN,INST_DPD_SUM,INST_DBD_MAX,INST_DBD_MEAN,INST_DBD_SUM,INST_LATE_PAYMENT_FLAG_MEAN,INST_LATE_PAYMENT_FLAG_SUM,INST_UNDERPAYMENT_FLAG_MEAN,INST_UNDERPAYMENT_FLAG_SUM
0,100001,2,2,4,2.714286,-2916.0,-1619.0,-2187.714355,-2916.0,-1628.0,...,11.0,1.571429,11.0,36.0,8.857142,62.0,0.142857,1,0.0,0
1,100002,1,2,19,10.000000,-565.0,-25.0,-295.000000,-587.0,-49.0,...,0.0,0.000000,0.0,31.0,20.421053,388.0,0.000000,0,0.0,0
2,100003,3,2,12,5.080000,-2310.0,-536.0,-1378.160034,-2324.0,-544.0,...,0.0,0.000000,0.0,14.0,7.160000,179.0,0.000000,0,0.0,0
3,100004,1,2,3,2.000000,-784.0,-724.0,-754.000000,-795.0,-727.0,...,0.0,0.000000,0.0,11.0,7.666667,23.0,0.000000,0,0.0,0
4,100005,1,2,9,5.000000,-706.0,-466.0,-586.000000,-736.0,-470.0,...,1.0,0.111111,1.0,37.0,23.666666,213.0,0.111111,1,0.0,0


## 7. POS/Cash-balance features


In [ ]:
def aggregate_pos_cash():
    pos = load_csv("POS_CASH_balance.csv")

    status_dummies = pd.get_dummies(
        pos["NAME_CONTRACT_STATUS"],
        prefix="POS_STATUS",
        dummy_na=True,
        dtype="uint8",
    )

    pos_encoded = pd.concat(
        [pos.drop(columns=["NAME_CONTRACT_STATUS"]), status_dummies],
        axis=1,
    )

    agg = {
        "SK_ID_PREV": ["nunique"],
        "MONTHS_BALANCE": ["min", "max", "mean", "count"],
        "CNT_INSTALMENT": ["min", "max", "mean"],
        "CNT_INSTALMENT_FUTURE": ["min", "max", "mean"],
        "SK_DPD": ["max", "mean", "sum"],
        "SK_DPD_DEF": ["max", "mean", "sum"],
    }

    for col in status_dummies.columns:
        agg[col] = ["mean", "sum"]

    pos_agg = pos_encoded.groupby("SK_ID_CURR").agg(agg)
    pos_agg = flatten_columns(pos_agg).reset_index()
    pos_agg = pos_agg.rename(
        columns={
            col: f"POS_{col}"
            for col in pos_agg.columns
            if col != "SK_ID_CURR"
        }
    )

    del pos, pos_encoded, status_dummies
    gc.collect()

    return reduce_memory_usage(pos_agg)


pos_agg = aggregate_pos_cash()
print("POS/Cash aggregate shape:", pos_agg.shape)
display(pos_agg.head())


POS_CASH_balance.csv: (10001358, 8)
Memory: 1137.25 MB -> 803.42 MB (29.4% reduction)
Memory: 60.14 MB -> 34.74 MB (42.2% reduction)
POS/Cash aggregate shape: (337252, 38)


,SK_ID_CURR,POS_SK_ID_PREV_NUNIQUE,POS_MONTHS_BALANCE_MIN,POS_MONTHS_BALANCE_MAX,POS_MONTHS_BALANCE_MEAN,POS_MONTHS_BALANCE_COUNT,POS_CNT_INSTALMENT_MIN,POS_CNT_INSTALMENT_MAX,POS_CNT_INSTALMENT_MEAN,POS_CNT_INSTALMENT_FUTURE_MIN,...,POS_POS_STATUS_DEMAND_MEAN,POS_POS_STATUS_DEMAND_SUM,POS_POS_STATUS_RETURNED TO THE STORE_MEAN,POS_POS_STATUS_RETURNED TO THE STORE_SUM,POS_POS_STATUS_SIGNED_MEAN,POS_POS_STATUS_SIGNED_SUM,POS_POS_STATUS_XNA_MEAN,POS_POS_STATUS_XNA_SUM,POS_POS_STATUS_NAN_MEAN,POS_POS_STATUS_NAN_SUM
0,100001,2,-96,-53,-72.555557,9,4.0,4.0,4.000000,0.0,...,0.0,0,0.0,0,0.000000,0,0.0,0,0.0,0
1,100002,1,-19,-1,-10.000000,19,24.0,24.0,24.000000,6.0,...,0.0,0,0.0,0,0.000000,0,0.0,0,0.0,0
2,100003,3,-77,-18,-43.785713,28,6.0,12.0,10.107142,0.0,...,0.0,0,0.0,0,0.000000,0,0.0,0,0.0,0
3,100004,1,-27,-24,-25.500000,4,3.0,4.0,3.750000,0.0,...,0.0,0,0.0,0,0.000000,0,0.0,0,0.0,0
4,100005,1,-25,-15,-20.000000,11,9.0,12.0,11.700000,0.0,...,0.0,0,0.0,0,0.090909,1,0.0,0,0.0,0


## 8. Credit-card-balance features


In [ ]:
def aggregate_credit_card():
    credit_card = load_csv("credit_card_balance.csv")

    credit_card["CC_BALANCE_LIMIT_RATIO"] = safe_divide(
        credit_card["AMT_BALANCE"],
        credit_card["AMT_CREDIT_LIMIT_ACTUAL"],
    )

    credit_card["CC_PAYMENT_RECEIVABLE_RATIO"] = safe_divide(
        credit_card["AMT_PAYMENT_TOTAL_CURRENT"],
        credit_card["AMT_TOTAL_RECEIVABLE"],
    )

    category_cols = credit_card.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    credit_card_encoded = pd.get_dummies(
        credit_card,
        columns=category_cols,
        dummy_na=True,
        dtype="uint8",
    )

    agg = {
        "SK_ID_PREV": ["nunique"],
        "MONTHS_BALANCE": ["min", "max", "mean", "count"],
        "AMT_BALANCE": ["min", "max", "mean", "sum"],
        "AMT_CREDIT_LIMIT_ACTUAL": ["min", "max", "mean"],
        "AMT_DRAWINGS_ATM_CURRENT": ["max", "mean", "sum"],
        "AMT_DRAWINGS_CURRENT": ["max", "mean", "sum"],
        "AMT_DRAWINGS_POS_CURRENT": ["max", "mean", "sum"],
        "AMT_INST_MIN_REGULARITY": ["max", "mean", "sum"],
        "AMT_PAYMENT_CURRENT": ["max", "mean", "sum"],
        "AMT_PAYMENT_TOTAL_CURRENT": ["max", "mean", "sum"],
        "AMT_RECEIVABLE_PRINCIPAL": ["max", "mean", "sum"],
        "AMT_TOTAL_RECEIVABLE": ["max", "mean", "sum"],
        "CNT_DRAWINGS_ATM_CURRENT": ["max", "mean", "sum"],
        "CNT_DRAWINGS_CURRENT": ["max", "mean", "sum"],
        "CNT_DRAWINGS_POS_CURRENT": ["max", "mean", "sum"],
        "CNT_INSTALMENT_MATURE_CUM": ["max", "mean"],
        "SK_DPD": ["max", "mean", "sum"],
        "SK_DPD_DEF": ["max", "mean", "sum"],
        "CC_BALANCE_LIMIT_RATIO": ["min", "max", "mean"],
        "CC_PAYMENT_RECEIVABLE_RATIO": ["min", "max", "mean"],
    }

    dummy_cols = [
        col for col in credit_card_encoded.columns
        if col not in credit_card.columns
    ]

    for col in dummy_cols:
        agg[col] = ["mean", "sum"]

    agg = {
        col: funcs
        for col, funcs in agg.items()
        if col in credit_card_encoded.columns
    }

    credit_card_agg = credit_card_encoded.groupby("SK_ID_CURR").agg(agg)
    credit_card_agg = flatten_columns(credit_card_agg).reset_index()
    credit_card_agg = credit_card_agg.rename(
        columns={
            col: f"CC_{col}"
            for col in credit_card_agg.columns
            if col != "SK_ID_CURR"
        }
    )

    del credit_card, credit_card_encoded
    gc.collect()

    return reduce_memory_usage(credit_card_agg)


credit_card_agg = aggregate_credit_card()
print("Credit-card aggregate shape:", credit_card_agg.shape)
display(credit_card_agg.head())


credit_card_balance.csv: (3840312, 23)
Memory: 875.69 MB -> 681.58 MB (22.2% reduction)
Memory: 46.42 MB -> 38.42 MB (17.2% reduction)
Credit-card aggregate shape: (103558, 76)


,SK_ID_CURR,CC_SK_ID_PREV_NUNIQUE,CC_MONTHS_BALANCE_MIN,CC_MONTHS_BALANCE_MAX,CC_MONTHS_BALANCE_MEAN,CC_MONTHS_BALANCE_COUNT,CC_AMT_BALANCE_MIN,CC_AMT_BALANCE_MAX,CC_AMT_BALANCE_MEAN,CC_AMT_BALANCE_SUM,...,CC_NAME_CONTRACT_STATUS_DEMAND_MEAN,CC_NAME_CONTRACT_STATUS_DEMAND_SUM,CC_NAME_CONTRACT_STATUS_REFUSED_MEAN,CC_NAME_CONTRACT_STATUS_REFUSED_SUM,CC_NAME_CONTRACT_STATUS_SENT PROPOSAL_MEAN,CC_NAME_CONTRACT_STATUS_SENT PROPOSAL_SUM,CC_NAME_CONTRACT_STATUS_SIGNED_MEAN,CC_NAME_CONTRACT_STATUS_SIGNED_SUM,CC_NAME_CONTRACT_STATUS_NAN_MEAN,CC_NAME_CONTRACT_STATUS_NAN_SUM
0,100006,1,-6,-1,-3.5,6,0.0,0.00,0.000000,0.000,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
1,100011,1,-75,-2,-38.5,74,0.0,189000.00,54482.111149,4031676.225,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
2,100013,1,-96,-1,-48.5,96,0.0,161420.22,18159.919219,1743352.245,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
3,100021,1,-18,-2,-10.0,17,0.0,0.00,0.000000,0.000,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0
4,100023,1,-11,-4,-7.5,8,0.0,0.00,0.000000,0.000,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0


## 9. Merge all aggregated tables

Every aggregated table must contain only one row per `SK_ID_CURR`.


In [ ]:
aggregated_tables = {
    "bureau": bureau_agg,
    "previous": previous_agg,
    "installments": installments_agg,
    "pos_cash": pos_agg,
    "credit_card": credit_card_agg,
}

for name, table in aggregated_tables.items():
    duplicate_count = table["SK_ID_CURR"].duplicated().sum()
    print(f"{name}: shape={table.shape}, duplicate IDs={duplicate_count}")


bureau: shape=(305811, 130), duplicate IDs=0
previous: shape=(338857, 358), duplicate IDs=0
installments: shape=(339587, 36), duplicate IDs=0
pos_cash: shape=(337252, 38), duplicate IDs=0
credit_card: shape=(103558, 76), duplicate IDs=0


In [ ]:
def merge_aggregates(application_df, tables):
    merged = application_df.copy()
    original_rows = len(merged)

    for name, table in tables.items():
        before_shape = merged.shape
        merged = merged.merge(
            table,
            on="SK_ID_CURR",
            how="left",
            validate="one_to_one",
        )

        print(
            f"Merged {name}: {before_shape} -> {merged.shape}"
        )

        if len(merged) != original_rows:
            raise ValueError(
                f"Row count changed after merging {name}."
            )

    return merged


train_merged = merge_aggregates(
    application_train,
    aggregated_tables,
)

test_merged = merge_aggregates(
    application_test,
    aggregated_tables,
)

print("Final train shape:", train_merged.shape)
print("Final test shape:", test_merged.shape)


Merged bureau: (307511, 147) -> (307511, 276)
Merged previous: (307511, 276) -> (307511, 633)
Merged installments: (307511, 633) -> (307511, 668)
Merged pos_cash: (307511, 668) -> (307511, 705)
Merged credit_card: (307511, 705) -> (307511, 780)
Merged bureau: (48744, 146) -> (48744, 275)
Merged previous: (48744, 275) -> (48744, 632)
Merged installments: (48744, 632) -> (48744, 667)
Merged pos_cash: (48744, 667) -> (48744, 704)
Merged credit_card: (48744, 704) -> (48744, 779)
Final train shape: (307511, 780)
Final test shape: (48744, 779)


## 10. Final cross-table engineered features

These features combine information from multiple source tables.


In [ ]:
def add_cross_table_features(df):
    df = df.copy()

    # Historical credit exposure relative to current annual income.
    if "BUREAU_AMT_CREDIT_SUM_SUM" in df.columns:
        df["BUREAU_CREDIT_INCOME_RATIO"] = safe_divide(
            df["BUREAU_AMT_CREDIT_SUM_SUM"],
            df["AMT_INCOME_TOTAL"],
        )

    if "PREV_AMT_CREDIT_SUM" in df.columns:
        df["PREVIOUS_CREDIT_INCOME_RATIO"] = safe_divide(
            df["PREV_AMT_CREDIT_SUM"],
            df["AMT_INCOME_TOTAL"],
        )

    # Historical installment payments relative to current annual income.
    if "INST_AMT_PAYMENT_SUM" in df.columns:
        df["HISTORICAL_PAYMENT_INCOME_RATIO"] = safe_divide(
            df["INST_AMT_PAYMENT_SUM"],
            df["AMT_INCOME_TOTAL"],
        )

    # Presence of data in each historical source.
    presence_columns = {
        "HAS_BUREAU_HISTORY": "BUREAU_SK_ID_BUREAU_COUNT",
        "HAS_PREVIOUS_APPLICATION": "PREV_SK_ID_PREV_COUNT",
        "HAS_INSTALLMENT_HISTORY": "INST_SK_ID_PREV_NUNIQUE",
        "HAS_POS_HISTORY": "POS_SK_ID_PREV_NUNIQUE",
        "HAS_CREDIT_CARD_HISTORY": "CC_SK_ID_PREV_NUNIQUE",
    }

    created_presence_features = []

    for feature_name, source_col in presence_columns.items():
        if source_col in df.columns:
            df[feature_name] = df[source_col].notna().astype("uint8")
            created_presence_features.append(feature_name)

    if created_presence_features:
        df["AVAILABLE_HISTORY_SOURCE_COUNT"] = df[
            created_presence_features
        ].sum(axis=1)

    return df


train_merged = add_cross_table_features(train_merged)
test_merged = add_cross_table_features(test_merged)

train_merged = reduce_memory_usage(train_merged)
test_merged = reduce_memory_usage(test_merged)

print("Train shape after cross-table features:", train_merged.shape)
print("Test shape after cross-table features:", test_merged.shape)


Memory: 1558.13 MB -> 1251.38 MB (19.7% reduction)
Memory: 246.97 MB -> 198.35 MB (19.7% reduction)
Train shape after cross-table features: (307511, 789)
Test shape after cross-table features: (48744, 788)


## 11. Validate train and test compatibility


In [ ]:
target = train_merged.pop("TARGET")

train_columns = set(train_merged.columns)
test_columns = set(test_merged.columns)

missing_in_test = sorted(train_columns - test_columns)
extra_in_test = sorted(test_columns - train_columns)

print("Columns missing in test:", len(missing_in_test))
print("Extra columns in test:", len(extra_in_test))

if missing_in_test:
    print("Examples missing in test:", missing_in_test[:10])

if extra_in_test:
    print("Examples extra in test:", extra_in_test[:10])

# Align the datasets to guarantee identical feature columns.
train_merged, test_merged = train_merged.align(
    test_merged,
    join="left",
    axis=1,
    fill_value=np.nan,
)

train_merged["TARGET"] = target.values

print("Aligned train shape:", train_merged.shape)
print("Aligned test shape:", test_merged.shape)


Columns missing in test: 0
Extra columns in test: 0
Aligned train shape: (307511, 789)
Aligned test shape: (48744, 788)


## 12. Basic data-quality report


In [ ]:
def data_quality_report(df, top_n=20):
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_percent": df.isna().mean() * 100,
        "unique_values": df.nunique(dropna=True),
    })

    return report.sort_values(
        "missing_percent",
        ascending=False,
    ).head(top_n)


display(data_quality_report(train_merged))


,dtype,missing_count,missing_percent,unique_values
CC_CC_PAYMENT_RECEIVABLE_RATIO_MIN,float64,247736,80.561671,9923
CC_CC_PAYMENT_RECEIVABLE_RATIO_MAX,float64,247736,80.561671,56919
CC_CC_PAYMENT_RECEIVABLE_RATIO_MEAN,float64,247736,80.561671,57349
CC_AMT_PAYMENT_CURRENT_MEAN,float64,246451,80.143800,56764
CC_AMT_PAYMENT_CURRENT_MAX,float64,246451,80.143800,25753
...,...,...,...,...
CC_CC_BALANCE_LIMIT_RATIO_MAX,float32,221475,72.021814,55724
CC_CC_BALANCE_LIMIT_RATIO_MEAN,float32,221475,72.021814,59047
CC_CC_BALANCE_LIMIT_RATIO_MIN,float32,221475,72.021814,12331
CC_CNT_INSTALMENT_MATURE_CUM_MAX,float32,220606,71.739222,120


In [ ]:
print("Duplicate SK_ID_CURR:", train_merged["SK_ID_CURR"].duplicated().sum())
print("Infinite values:", np.isinf(
    train_merged.select_dtypes(include=np.number)
).sum().sum())

print("\nTarget distribution:")
display(
    train_merged["TARGET"]
    .value_counts(normalize=True)
    .rename("proportion")
)


Duplicate SK_ID_CURR: 0
Infinite values: 0

Target distribution:


TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

## 13. Prepare `X`, `y`, and `X_test`

Keep `SK_ID_CURR` separately for submissions. It should generally not be used as a model feature.


In [ ]:
train_ids = train_merged["SK_ID_CURR"].copy()
test_ids = test_merged["SK_ID_CURR"].copy()

X = train_merged.drop(columns=["TARGET", "SK_ID_CURR"])
y = train_merged["TARGET"].astype("int8")
X_test = test_merged.drop(columns=["SK_ID_CURR"])

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)

print("\nFeature types:")
print(X.dtypes.value_counts())


X shape: (307511, 787)
y shape: (307511,)
X_test shape: (48744, 787)

Feature types:
float32    635
float64     86
int8        48
object      16
int16        2
Name: count, dtype: int64


In [ ]:
len(X.select_dtypes(include=["object"]).columns)

16

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print("Shape of X_train",X_train.shape)
print("Shape of y_train",y_train.shape)
print("Shape of X_val",X_val.shape)
print("Shape of y_val",y_val.shape)

Shape of X_train (246008, 787)
Shape of y_train (246008,)
Shape of X_val (61503, 787)
Shape of y_val (61503,)


In [ ]:
categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

In [ ]:
X_enc = X.copy()
y_enc = y.copy()

#### **Missing value Imputation**

In [ ]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

X_train[numerical_cols] = num_imputer.fit_transform(
    X_train[numerical_cols]
)

X_val[numerical_cols] = num_imputer.transform(
    X_val[numerical_cols]
)

X_test[numerical_cols] = num_imputer.transform(
    X_test[numerical_cols]
)

X_train[categorical_cols] = cat_imputer.fit_transform(
    X_train[categorical_cols]
)

X_val[categorical_cols] = cat_imputer.transform(
    X_val[categorical_cols]
)

X_test[categorical_cols] = cat_imputer.transform(
    X_test[categorical_cols]
)

In [ ]:
X_enc[numerical_cols] = num_imputer.fit_transform(
    X_enc[numerical_cols]
)

X_enc[categorical_cols] = cat_imputer.fit_transform(
    X_enc[categorical_cols]
)

In [ ]:
import inspect
import app.predictor as predictor

print(predictor.__file__)
print(inspect.signature(predictor.predict_default_probability))

/Users/jayeshadwani/ml-learning/notebooks/Case Studies/Credit Risk Prediction/home_credit_risk_api/app/predictor.py
(records: list[dict[str, typing.Any]], include_explanation: bool = False) -> list[dict[str, typing.Any]]


In [ ]:
import pandas as pd

from app.predictor import predict_default_probability


validation_records = X_val.to_dict(orient="records")

predictions = predict_default_probability(
    records=validation_records,
    include_explanation=False,
)

prediction_df = pd.DataFrame(predictions)

demo_candidates = X_val.reset_index(drop=True).copy()
demo_candidates["actual_default"] = (
    pd.Series(y_val).reset_index(drop=True)
)

demo_candidates["default_probability"] = (
    prediction_df["default_probability"]
)

demo_candidates["risk_category"] = (
    prediction_df["risk_category"]
)

In [ ]:
sampling_plan = {
    "low": 6,
    "medium": 6,
    "high": 8,
}

selected_groups = []

for risk_category, sample_size in sampling_plan.items():
    group = demo_candidates[
        demo_candidates["risk_category"] == risk_category
    ]

    sampled_group = group.sample(
        n=min(sample_size, len(group)),
        random_state=42,
    )

    selected_groups.append(sampled_group)

demo_applicants = (
    pd.concat(selected_groups)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print(
    demo_applicants[
        [
            "actual_default",
            "default_probability",
            "risk_category",
        ]
    ].groupby(
        ["risk_category", "actual_default"]
    ).size()
)

print("Total demo applicants:", len(demo_applicants))

risk_category  actual_default
high           0                 8
low            0                 6
medium         0                 6
dtype: int64
Total demo applicants: 20


In [ ]:
print("Overall validation targets:")
print(demo_candidates["actual_default"].value_counts())

print("\nDefaults available by risk category:")
print(
    demo_candidates
    .groupby(["risk_category", "actual_default"])
    .size()
)

Overall validation targets:
actual_default
0    56538
1     4965
Name: count, dtype: int64

Defaults available by risk category:
risk_category  actual_default
high           0                  9388
               1                  2853
low            0                 30165
               1                   715
medium         0                 16985
               1                  1397
dtype: int64


In [ ]:
sampling_plan = {
    ("high", 1): 5,
    ("high", 0): 3,
    ("medium", 1): 3,
    ("medium", 0): 3,
    ("low", 1): 2,
    ("low", 0): 4,
}

selected_groups = []

for (risk_category, actual_default), sample_size in sampling_plan.items():

    group = demo_candidates[
        (demo_candidates["risk_category"] == risk_category)
        & (demo_candidates["actual_default"] == actual_default)
    ]

    sampled_group = group.sample(
        n=sample_size,
        random_state=42,
    )

    selected_groups.append(sampled_group)


demo_applicants = (
    pd.concat(selected_groups, ignore_index=True)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

In [ ]:
print(
    demo_applicants
    .groupby(["risk_category", "actual_default"])
    .size()
)

print("\nTarget distribution:")
print(demo_applicants["actual_default"].value_counts())

print("\nTotal applicants:", len(demo_applicants))

risk_category  actual_default
high           0                 3
               1                 5
low            0                 4
               1                 2
medium         0                 3
               1                 3
dtype: int64

Target distribution:
actual_default
1    10
0    10
Name: count, dtype: int64

Total applicants: 20


In [ ]:
import json
from pathlib import Path

import pandas as pd

from app.predictor import feature_columns


OUTPUT_DIR = Path("demo_cases")
OUTPUT_DIR.mkdir(exist_ok=True)

demo_applicants = demo_applicants.copy()

demo_applicants["demo_case_id"] = [
    f"DEMO-{index:03d}"
    for index in range(1, len(demo_applicants) + 1)
]


# Convert applicant model inputs into JSON-safe records.
feature_records = json.loads(
    demo_applicants[feature_columns].to_json(
        orient="records"
    )
)


manifest_rows = []

for row_index, feature_record in enumerate(feature_records):

    row = demo_applicants.iloc[row_index]
    demo_case_id = row["demo_case_id"]

    case_directory = OUTPUT_DIR / demo_case_id
    case_directory.mkdir(exist_ok=True)

    applicant_payload = {
        "demo_case_id": demo_case_id,
        "applicant_features": feature_record,
    }

    with open(
        case_directory / "applicant_features.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            applicant_payload,
            file,
            indent=2,
            ensure_ascii=False,
        )

    manifest_rows.append({
        "demo_case_id": demo_case_id,
        "actual_default": int(row["actual_default"]),
        "default_probability": float(
            row["default_probability"]
        ),
        "risk_category": row["risk_category"],
    })


manifest_df = pd.DataFrame(manifest_rows)

manifest_df.to_csv(
    OUTPUT_DIR / "manifest.csv",
    index=False,
)

print(manifest_df.head())
print(f"\nSaved {len(manifest_df)} demo cases.")

  demo_case_id  actual_default  default_probability risk_category
0     DEMO-001               1             0.717420          high
1     DEMO-002               0             0.078169           low
2     DEMO-003               1             0.227351           low
3     DEMO-004               1             0.605462          high
4     DEMO-005               1             0.504279        medium

Saved 20 demo cases.


In [ ]:
import json
from pathlib import Path

import pandas as pd


DEMO_CASES_DIR = Path("demo_cases")

manifest_df = pd.read_csv(
    DEMO_CASES_DIR / "manifest.csv"
).set_index("demo_case_id")


def safe_float(value):
    if value is None or pd.isna(value):
        return None
    return float(value)


for case_directory in DEMO_CASES_DIR.iterdir():

    if not case_directory.is_dir():
        continue

    case_id = case_directory.name

    features_path = (
        case_directory / "applicant_features.json"
    )

    with open(features_path, "r", encoding="utf-8") as file:
        payload = json.load(file)

    features = payload["applicant_features"]

    income = safe_float(
        features.get("AMT_INCOME_TOTAL")
    )

    credit_amount = safe_float(
        features.get("AMT_CREDIT")
    )

    annuity = safe_float(
        features.get("AMT_ANNUITY")
    )

    days_birth = safe_float(
        features.get("DAYS_BIRTH")
    )

    days_employed = safe_float(
        features.get("DAYS_EMPLOYED")
    )

    age_years = (
        round(abs(days_birth) / 365.25, 1)
        if days_birth is not None
        else None
    )

    employment_years = None

    if (
        days_employed is not None
        and days_employed != 365243
    ):
        employment_years = round(
            abs(days_employed) / 365.25,
            1,
        )

    credit_income_ratio = (
        round(credit_amount / income, 3)
        if income and credit_amount is not None
        else None
    )

    annuity_income_ratio = (
        round(annuity / income, 3)
        if income and annuity is not None
        else None
    )

    manifest_row = manifest_df.loc[case_id]

    summary = {
        "demo_case_id": case_id,

        "application": {
            "contract_type": features.get(
                "NAME_CONTRACT_TYPE"
            ),
            "credit_amount": credit_amount,
            "loan_annuity": annuity,
            "goods_price": safe_float(
                features.get("AMT_GOODS_PRICE")
            ),
        },

        "financial_profile": {
            "total_income": income,
            "income_type": features.get(
                "NAME_INCOME_TYPE"
            ),
            "credit_income_ratio": credit_income_ratio,
            "annuity_income_ratio": annuity_income_ratio,
        },

        "employment_profile": {
            "occupation_type": features.get(
                "OCCUPATION_TYPE"
            ),
            "organization_type": features.get(
                "ORGANIZATION_TYPE"
            ),
            "employment_years": employment_years,
        },

        "household_profile": {
            "family_status": features.get(
                "NAME_FAMILY_STATUS"
            ),
            "housing_type": features.get(
                "NAME_HOUSING_TYPE"
            ),
            "children_count": safe_float(
                features.get("CNT_CHILDREN")
            ),
            "family_members": safe_float(
                features.get("CNT_FAM_MEMBERS")
            ),
        },

        "credit_indicators": {
            "external_score_1": safe_float(
                features.get("EXT_SOURCE_1")
            ),
            "external_score_2": safe_float(
                features.get("EXT_SOURCE_2")
            ),
            "external_score_3": safe_float(
                features.get("EXT_SOURCE_3")
            ),
        },

        "model_output": {
            "default_probability": float(
                manifest_row["default_probability"]
            ),
            "risk_category": manifest_row[
                "risk_category"
            ],
        },

        "metadata": {
            "age_years": age_years,
            "sensitive_features_excluded": [
                "CODE_GENDER"
            ],
        },
    }

    with open(
        case_directory / "applicant_summary.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            summary,
            file,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )


print("Applicant summaries generated successfully.")

Applicant summaries generated successfully.


In [ ]:
with open(
    "demo_cases/DEMO-001/applicant_summary.json",
    encoding="utf-8",
) as file:
    summary = json.load(file)

print(json.dumps(summary, indent=2))

{
  "demo_case_id": "DEMO-001",
  "application": {
    "contract_type": "Cash loans",
    "credit_amount": 271066.5,
    "loan_annuity": 24988.5,
    "goods_price": 234000.0
  },
  "financial_profile": {
    "total_income": 126000.0,
    "income_type": "Working",
    "credit_income_ratio": 2.151,
    "annuity_income_ratio": 0.198
  },
  "employment_profile": {
    "occupation_type": "Laborers",
    "organization_type": "Transport: type 4",
    "employment_years": 5.8
  },
  "household_profile": {
    "family_status": "Civil marriage",
    "housing_type": "House / apartment",
    "children_count": 0.0,
    "family_members": 2.0
  },
  "credit_indicators": {
    "external_score_1": 0.166307807,
    "external_score_2": 0.1157590821,
    "external_score_3": 0.5352762341
  },
  "model_output": {
    "default_probability": 0.7174199819564819,
    "risk_category": "high"
  },
  "metadata": {
    "age_years": 26.3,
    "sensitive_features_excluded": [
      "CODE_GENDER"
    ]
  }
}


In [ ]:
import json
from pathlib import Path

from app.predictor import predict_default_probability


DEMO_CASES_DIR = Path("demo_cases")

SENSITIVE_FEATURES = {
    "CODE_GENDER",
}


for case_directory in DEMO_CASES_DIR.iterdir():

    if not case_directory.is_dir():
        continue

    features_path = (
        case_directory / "applicant_features.json"
    )

    with open(features_path, "r", encoding="utf-8") as file:
        payload = json.load(file)

    case_id = payload["demo_case_id"]
    applicant_features = payload["applicant_features"]

    prediction = predict_default_probability(
        records=[applicant_features],
        include_explanation=True,
    )[0]

    # Remove sensitive attributes from GenAI-facing explanations.
    risk_factors = [
        factor
        for factor in prediction["top_risk_factors"]
        if factor["feature"] not in SENSITIVE_FEATURES
    ][:3]

    protective_factors = [
        factor
        for factor in prediction["top_protective_factors"]
        if factor["feature"] not in SENSITIVE_FEATURES
    ][:2]

    risk_explanation = {
        "demo_case_id": case_id,

        "model_output": {
            "default_probability": prediction[
                "default_probability"
            ],
            "non_default_probability": prediction[
                "non_default_probability"
            ],
            "risk_category": prediction["risk_category"],
            "model_version": prediction["model_version"],
        },

        "top_risk_factors": risk_factors,
        "top_protective_factors": protective_factors,

        "explanation_metadata": {
            "method": "SHAP",
            "shap_output_scale": prediction[
                "shap_output_scale"
            ],
            "sensitive_features_excluded": list(
                SENSITIVE_FEATURES
            ),
        },
    }

    output_path = (
        case_directory / "risk_explanation.json"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            risk_explanation,
            file,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )


print("Risk explanation files generated successfully.")

Risk explanation files generated successfully.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


DEMO_CASES_DIR = Path("demo_cases")

SENSITIVE_FEATURES = {
    "CODE_GENDER",
}


manifest_df = pd.read_csv(
    DEMO_CASES_DIR / "manifest.csv"
).set_index("demo_case_id")


validation_errors = []


for case_directory in sorted(DEMO_CASES_DIR.iterdir()):

    if not case_directory.is_dir():
        continue

    case_id = case_directory.name

    required_files = [
        "applicant_features.json",
        "applicant_summary.json",
        "risk_explanation.json",
    ]

    for filename in required_files:
        if not (case_directory / filename).exists():
            validation_errors.append(
                f"{case_id}: missing {filename}"
            )

    if any(
        not (case_directory / filename).exists()
        for filename in required_files
    ):
        continue

    with open(
        case_directory / "applicant_features.json",
        encoding="utf-8",
    ) as file:
        features_payload = json.load(file)

    with open(
        case_directory / "applicant_summary.json",
        encoding="utf-8",
    ) as file:
        summary = json.load(file)

    with open(
        case_directory / "risk_explanation.json",
        encoding="utf-8",
    ) as file:
        explanation = json.load(file)

    # Case IDs must match.
    if features_payload["demo_case_id"] != case_id:
        validation_errors.append(
            f"{case_id}: features case ID mismatch"
        )

    if summary["demo_case_id"] != case_id:
        validation_errors.append(
            f"{case_id}: summary case ID mismatch"
        )

    if explanation["demo_case_id"] != case_id:
        validation_errors.append(
            f"{case_id}: explanation case ID mismatch"
        )

    # Target must not appear in GenAI-facing files.
    summary_text = json.dumps(summary)
    explanation_text = json.dumps(explanation)

    if "actual_default" in summary_text:
        validation_errors.append(
            f"{case_id}: target leaked into summary"
        )

    if "actual_default" in explanation_text:
        validation_errors.append(
            f"{case_id}: target leaked into explanation"
        )

    # Sensitive features must not appear in explanations.
    explanation_features = {
        factor["feature"]
        for factor in (
            explanation["top_risk_factors"]
            + explanation["top_protective_factors"]
        )
    }

    leaked_sensitive_features = (
        explanation_features & SENSITIVE_FEATURES
    )

    if leaked_sensitive_features:
        validation_errors.append(
            f"{case_id}: sensitive features leaked: "
            f"{leaked_sensitive_features}"
        )

    # Check explanation sizes.
    if len(explanation["top_risk_factors"]) > 3:
        validation_errors.append(
            f"{case_id}: more than 3 risk factors"
        )

    if len(explanation["top_protective_factors"]) > 2:
        validation_errors.append(
            f"{case_id}: more than 2 protective factors"
        )

    model_output = explanation["model_output"]

    default_probability = model_output[
        "default_probability"
    ]

    non_default_probability = model_output[
        "non_default_probability"
    ]

    if not np.isclose(
        default_probability + non_default_probability,
        1.0,
    ):
        validation_errors.append(
            f"{case_id}: probabilities do not sum to 1"
        )

    # Compare against manifest.
    manifest_row = manifest_df.loc[case_id]

    if not np.isclose(
        default_probability,
        manifest_row["default_probability"],
    ):
        validation_errors.append(
            f"{case_id}: probability differs from manifest"
        )

    if (
        model_output["risk_category"]
        != manifest_row["risk_category"]
    ):
        validation_errors.append(
            f"{case_id}: risk category differs from manifest"
        )


if validation_errors:
    print("Validation failed:\n")

    for error in validation_errors:
        print("-", error)

else:
    print("All 20 demo cases passed validation.")

All 20 demo cases passed validation.


In [ ]:
with open(
    "demo_cases/DEMO-001/risk_explanation.json",
    encoding="utf-8",
) as file:
    explanation = json.load(file)

print(json.dumps(explanation, indent=2))

{
  "demo_case_id": "DEMO-001",
  "model_output": {
    "default_probability": 0.7174199819564819,
    "non_default_probability": 0.28258001804351807,
    "risk_category": "high",
    "model_version": "1.1.0"
  },
  "top_risk_factors": [
    {
      "feature": "EXT_SOURCE_MEAN",
      "display_name": "Ext Source Mean",
      "raw_value": 0.1410334408,
      "model_value": 0.1410334408,
      "shap_value": 0.798836350440979,
      "impact": "increases_default_risk"
    },
    {
      "feature": "CREDIT_ANNUITY_RATIO",
      "display_name": "Credit Annuity Ratio",
      "raw_value": 10.8476495743,
      "model_value": 10.8476495743,
      "shap_value": 0.13348932564258575,
      "impact": "increases_default_risk"
    },
    {
      "feature": "INST_LATE_PAYMENT_FLAG_MEAN",
      "display_name": "Inst Late Payment Flag Mean",
      "raw_value": 0.2093023211,
      "model_value": 0.2093023211,
      "shap_value": 0.10084374994039536,
      "impact": "increases_default_risk"
    }
  ],
  "t

In [ ]:
assert explanation["demo_case_id"] == "DEMO-001"
assert "CODE_GENDER" not in {
    factor["feature"]
    for factor in explanation["top_risk_factors"]
}

In [ ]:
import sys

print(sys.executable)

/opt/anaconda3/envs/home_credit_api/bin/python


#### **Encoding Categorical features**

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train[categorical_cols] = encoder.fit_transform(
    X_train[categorical_cols]
)

X_val[categorical_cols] = encoder.transform(
    X_val[categorical_cols]
)

X_test[categorical_cols] = encoder.transform(
    X_test[categorical_cols]
)

In [ ]:
X_enc[categorical_cols] = encoder.fit_transform(
    X_enc[categorical_cols]
)

In [ ]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print(scale_pos_weight)

11.38710976837865


In [61]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

log_model = LogisticRegression(
    penalty="l2",
    C=1.0,
    class_weight={
        0: 1,
        1: scale_pos_weight
    },
    solver="saga",
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

log_model.fit(X_train, y_train)

train_prob = log_model.predict_proba(X_train)[:, 1]
val_prob = log_model.predict_proba(X_val)[:, 1]

print("Train ROC-AUC:", roc_auc_score(y_train, train_prob))
print("Validation ROC-AUC:", roc_auc_score(y_val, val_prob))

Train ROC-AUC: 0.6470383158256157
Validation ROC-AUC: 0.6491576626608767


In [62]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

rf_model = RandomForestClassifier(
    n_estimators=350,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    class_weight={
        0: 1,
        1: scale_pos_weight
    },
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

train_prob = rf_model.predict_proba(X_train)[:, 1]
val_prob = rf_model.predict_proba(X_val)[:, 1]

print("Train ROC-AUC:", roc_auc_score(y_train, train_prob))
print("Validation ROC-AUC:", roc_auc_score(y_val, val_prob))

Train ROC-AUC: 0.8960125362654524
Validation ROC-AUC: 0.7632571942185273


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score

rf_model = RandomForestClassifier(
    class_weight={
        0: 1,
        1: scale_pos_weight
    },
    random_state=42,
    n_jobs=1
)

param_distributions = {
    "n_estimators": [200, 300, 400, 500],
    "max_depth": [8, 12, 16, 20, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": ["sqrt", "log2", 0.5],
    "max_samples": [0.7, 0.8, 0.9, None],
    "bootstrap": [True]
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="roc_auc",
    cv=cv,
    refit=True,
    random_state=42,
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

random_search.fit(X_train, y_train)

best_rf_model = random_search.best_estimator_

train_prob = best_rf_model.predict_proba(X_train)[:, 1]
val_prob = best_rf_model.predict_proba(X_val)[:, 1]

print("Best parameters:")
print(random_search.best_params_)

print("\nBest CV ROC-AUC:")
print(round(random_search.best_score_, 4))

print("\nTrain ROC-AUC:")
print(round(roc_auc_score(y_train, train_prob), 4))

print("Validation ROC-AUC:")
print(round(roc_auc_score(y_val, val_prob), 4))

Fitting 3 folds for each of 20 candidates, totalling 60 fits


In [ ]:
aaa

In [ ]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=350,
    max_depth=3,
    learning_rate=0.1,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0,
    reg_lambda=3,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train,
    eval_set=[
        (X_train, y_train),
        (X_val, y_val)
    ],
    verbose=50
)

[0]	validation_0-auc:0.71539	validation_1-auc:0.71335
[50]	validation_0-auc:0.77065	validation_1-auc:0.76715
[100]	validation_0-auc:0.78593	validation_1-auc:0.77814
[150]	validation_0-auc:0.79441	validation_1-auc:0.78281
[200]	validation_0-auc:0.80059	validation_1-auc:0.78535
[250]	validation_0-auc:0.80615	validation_1-auc:0.78678
[300]	validation_0-auc:0.81044	validation_1-auc:0.78730
[349]	validation_0-auc:0.81469	validation_1-auc:0.78830


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=1, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=350, n_jobs=-1,
              num_parallel_tree=None, random_state=42, ...)

In [ ]:
from sklearn.metrics import roc_auc_score

train_prob = model.predict_proba(X_train)[:, 1]
val_prob = model.predict_proba(X_val)[:, 1]

train_auc = roc_auc_score(y_train, train_prob)
val_auc = roc_auc_score(y_val, val_prob)

print("Train ROC-AUC:", train_auc)
print("Validation ROC-AUC:", val_auc)

Train ROC-AUC: 0.8146866264944161
Validation ROC-AUC: 0.7882957881583409


In [ ]:
import numpy as np
import pandas as pd


# Generate validation probabilities
validation_probabilities = model.predict_proba(X_val)[:, 1]


# Bottom 50% = low
# Next 30% = medium
# Top 20% = high
medium_risk_cutoff = float(
    np.quantile(validation_probabilities, 0.50)
)

high_risk_cutoff = float(
    np.quantile(validation_probabilities, 0.80)
)


print("Medium-risk cutoff:", medium_risk_cutoff)
print("High-risk cutoff:", high_risk_cutoff)

Medium-risk cutoff: 0.34990549087524414
High-risk cutoff: 0.5973379611968994


In [ ]:
validation_results = pd.DataFrame({
    "actual_default": np.asarray(y_val),
    "default_probability": validation_probabilities,
})


def assign_risk_category(probability: float) -> str:
    if probability >= high_risk_cutoff:
        return "high"

    if probability >= medium_risk_cutoff:
        return "medium"

    return "low"


validation_results["risk_category"] = (
    validation_results["default_probability"]
    .apply(assign_risk_category)
)


risk_summary = (
    validation_results
    .groupby("risk_category")
    .agg(
        applicants=("actual_default", "size"),
        defaulters=("actual_default", "sum"),
        bad_rate=("actual_default", "mean"),
        average_probability=("default_probability", "mean"),
    )
)


risk_summary["population_share"] = (
    risk_summary["applicants"]
    / len(validation_results)
)

risk_summary["default_capture_rate"] = (
    risk_summary["defaulters"]
    / validation_results["actual_default"].sum()
)

overall_bad_rate = validation_results["actual_default"].mean()

risk_summary["bad_rate_lift"] = (
    risk_summary["bad_rate"]
    / overall_bad_rate
)


risk_summary = risk_summary.reindex([
    "low",
    "medium",
    "high",
])

display(risk_summary)

,applicants,defaulters,bad_rate,average_probability,population_share,default_capture_rate,bad_rate_lift
risk_category,,,,,,,
low,30751,709,0.023056,0.205564,0.499992,0.142800,0.285604
medium,18451,1399,0.075822,0.463440,0.300002,0.281772,0.939236
high,12301,2857,0.232258,0.728638,0.200007,0.575428,2.877046


In [ ]:
pd.DataFrame({
    "weights": model.feature_importances_,
    "columns": X_train.columns
}).sort_values(by="weights",ascending=False).head(10)

,weights,columns
133,0.066472,EXT_SOURCE_MEAN
135,0.019360,EXT_SOURCE_MAX
11,0.011824,NAME_EDUCATION_TYPE
662,0.011101,INST_LATE_PAYMENT_FLAG_MEAN
128,0.011070,CREDIT_GOODS_RATIO
740,0.009793,CC_CNT_DRAWINGS_ATM_CURRENT_MEAN
397,0.009778,PREV_NAME_CONTRACT_STATUS_APPROVED_MEAN
182,0.009206,BUREAU_BUREAU_DEBT_CREDIT_RATIO_MEAN
401,0.008531,PREV_NAME_CONTRACT_STATUS_REFUSED_MEAN
137,0.008383,EXT_SOURCE_MISSING_COUNT


In [ ]:
from pathlib import Path

import joblib


BUNDLE_PATH = Path("artifacts/home_credit_bundle.joblib")

bundle = joblib.load(BUNDLE_PATH)

bundle["risk_thresholds"] = {
    "medium": 0.34990549087524414,
    "high": 0.5973379611968994,
}

bundle["risk_band_definition"] = {
    "low_population_share": 0.50,
    "medium_population_share": 0.30,
    "high_population_share": 0.20,
}

bundle["model_version"] = "1.1.0"

joblib.dump(bundle, BUNDLE_PATH)

print("Updated bundle saved successfully.")

Updated bundle saved successfully.


In [ ]:
updated_bundle = joblib.load(BUNDLE_PATH)

print(updated_bundle["risk_thresholds"])
print(updated_bundle["model_version"])

{'medium': 0.34990549087524414, 'high': 0.5973379611968994}
1.1.0


In [ ]:
import pandas as pd

from app.predictor import predict_default_probability


validation_records = X_val_raw.to_dict(orient="records")

predictions = predict_default_probability(
    records=validation_records,
    include_explanation=False,
)

prediction_df = pd.DataFrame(predictions)

demo_candidates = X_val_raw.reset_index(drop=True).copy()
demo_candidates["actual_default"] = (
    pd.Series(y_val).reset_index(drop=True)
)

demo_candidates["default_probability"] = (
    prediction_df["default_probability"]
)

demo_candidates["risk_category"] = (
    prediction_df["risk_category"]
)

## 14. Save merged datasets

Parquet is recommended because it is faster and smaller than CSV. If Parquet support is unavailable, install `pyarrow` or save as CSV.


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    model,
    X_enc,
    y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print("Fold ROC-AUC:", scores)
print("Mean ROC-AUC:", scores.mean())
print("Standard deviation:", scores.std())

KeyboardInterrupt: 

In [ ]:
OUTPUT_DIR = Path("/content/home_credit_processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_output = OUTPUT_DIR / "home_credit_train_merged.parquet"
test_output = OUTPUT_DIR / "home_credit_test_merged.parquet"

#train_merged.to_parquet(train_output, index=False)
#test_merged.to_parquet(test_output, index=False)

print("Saved:")
print(train_output)
print(test_output)


OSError: [Errno 30] Read-only file system: '/content'

In [ ]:
# Uncomment when CSV output is required.

# train_merged.to_csv(
#     OUTPUT_DIR / "home_credit_train_merged.csv",
#     index=False,
# )
#
# test_merged.to_csv(
#     OUTPUT_DIR / "home_credit_test_merged.csv",
#     index=False,
# )


## Notes for modeling

1. Split the training data before fitting encoders, imputers, or feature selectors.
2. Fit preprocessing only on the training fold to prevent leakage.
3. Keep `SK_ID_CURR` out of the model.
4. Use stratified cross-validation because `TARGET` is imbalanced.
5. Evaluate using ROC-AUC rather than accuracy alone.
6. Tree-based models such as LightGBM, XGBoost, CatBoost, and Random Forest can work with the engineered numeric features.
7. Categorical variables still need suitable encoding unless the selected model handles them natively.


In [ ]:
import joblib
import platform
import sklearn
import xgboost

model_bundle = {
    "model": model,
    "num_imputer": num_imputer,
    "cat_imputer": cat_imputer,
    "ordinal_encoder": encoder,
    "numerical_cols": numerical_cols,
    "categorical_cols": categorical_cols,
    "feature_columns": X_train.columns.tolist(),
    "metadata": {
        "model_name": "home_credit_xgboost",
        "cv_roc_auc_mean": 0.786595551011142,
        "cv_roc_auc_std": 0.004228946966212543,
        "python_version": platform.python_version(),
        "sklearn_version": sklearn.__version__,
        "xgboost_version": xgboost.__version__,
    },
}

joblib.dump(model_bundle, "home_credit_bundle.joblib")

['home_credit_bundle.joblib']

In [ ]:
loaded_bundle = joblib.load("home_credit_bundle.joblib")

original_prediction = model.predict_proba(
    X_val.iloc[[0]]
)[:, 1]

loaded_prediction = loaded_bundle["model"].predict_proba(
    X_val.iloc[[0]]
)[:, 1]

print("Original:", original_prediction)
print("Loaded:", loaded_prediction)

assert np.allclose(
    original_prediction,
    loaded_prediction,
    atol=1e-8,
)

print("Model saved correctly")

Original: [0.33924934]
Loaded: [0.33924934]
Model saved correctly


In [ ]:
import pandas
import numpy
import sklearn
import xgboost
import joblib

print("pandas==", pandas.__version__)
print("numpy==", numpy.__version__)
print("scikit-learn==", sklearn.__version__)
print("xgboost==", xgboost.__version__)
print("joblib==", joblib.__version__)

pandas== 2.3.3
numpy== 2.0.1
scikit-learn== 1.6.1
xgboost== 2.1.4
joblib== 1.5.2


In [ ]:
import json
import numpy as np

sample_records = (
    X_enc.iloc[[0]]
    .replace({np.nan: None})
    .to_dict(orient="records")
)

payload = {
    "records": sample_records
}

with open("sample_request.json", "w") as file:
    json.dump(payload, file, indent=2)

print("sample_request.json created")

sample_request.json created


In [ ]:
import os

print(os.getcwd())
print(os.path.abspath("sample_request.json"))

/Users/jayeshadwani/ml-learning/notebooks/Case Studies/Credit Risk Prediction
/Users/jayeshadwani/ml-learning/notebooks/Case Studies/Credit Risk Prediction/sample_request.json


In [ ]:
import numpy as np

times_ms = np.loadtxt("single_times.txt") * 1000

print("Median:", np.median(times_ms), "ms")
print("P95:", np.percentile(times_ms, 95), "ms")
print("P99:", np.percentile(times_ms, 99), "ms")

In [ ]:
import numpy as np

batch_times = np.loadtxt("batch_times.txt")
median_seconds = np.median(batch_times)

applications_per_second = 100 / median_seconds

print("Median batch time:", median_seconds, "seconds")
print("Throughput:", applications_per_second, "applications/second")

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_predict

oof_probability = cross_val_predict(
    model,
    X_enc,
    y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

y_array = np.asarray(y)

review_rate = 0.20
review_count = int(np.ceil(len(y_array) * review_rate))

highest_risk_indices = np.argsort(oof_probability)[::-1][:review_count]

defaults_captured = (
    y_array[highest_risk_indices].sum() / y_array.sum()
)

reviewed_bad_rate = y_array[highest_risk_indices].mean()
overall_bad_rate = y_array.mean()
lift = reviewed_bad_rate / overall_bad_rate

print("Review rate:", review_rate)
print("Defaults captured:", defaults_captured)
print("Bad-rate lift:", lift)

Review rate: 0.2
Defaults captured: 0.5676535750251762
Bad-rate lift: 2.838230956369071
